1 — load and parse:

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw_pos_export.csv")

# parse each row's timestamp format individually, instead of letting pandas
# infer one format for the whole column (which fails on mixed formats)
df["ts_parsed"] = pd.to_datetime(df["transaction_ts"], format="mixed", errors="coerce")

# sort so rows belonging to the same real transaction end up adjacent
df = df.sort_values(["store_id", "register_id", "ts_parsed"]).reset_index(drop=True)

# check: shape should be (131, 18) after adding ts_parsed, and NaT count
# should be 1 (the single blank transaction_ts) -- not more
print("shape:", df.shape)
print("unparsed timestamps (NaT):", df["ts_parsed"].isna().sum())

shape: (131, 18)
unparsed timestamps (NaT): 1


2- reconstruct transaction IDs:

In [2]:
GAP_MINUTES = 2  # max time between line items to count as the same transaction

# time since the previous line item for the same store + register + customer.
# dropna=False keeps walk-ins (blank customer_ref) as their own valid group,
# instead of pandas silently excluding them from the grouping.
df["prev_ts"] = df.groupby(["store_id", "register_id", "customer_ref"], dropna=False)["ts_parsed"].shift(1)
df["gap_minutes"] = (df["ts_parsed"] - df["prev_ts"]).dt.total_seconds() / 60

# a row starts a new transaction if there's no valid previous row, or too
# much time passed since the last one
df["new_transaction"] = df["prev_ts"].isna() | (df["gap_minutes"] > GAP_MINUTES)

# running count of "new transaction" flags gives every line item a
# transaction id — NOT using invoice_number as the key, since it's missing
# most of the time and can even repeat (see known edge cases below)
df["reconstructed_txn_id"] = df["new_transaction"].cumsum()

# a transaction is flagged for manual review if any of its lines had no
# parseable timestamp — those rows can't be trusted to sort into the right
# position, so their assigned id may be wrong
df["needs_manual_review"] = df["ts_parsed"].isna()

# check: how many distinct transactions did we get, and how many line
# items were flagged as needing review at the row level
print("distinct reconstructed transactions:", df["reconstructed_txn_id"].nunique())
print("rows flagged needs_manual_review:", df["needs_manual_review"].sum())

distinct reconstructed transactions: 63
rows flagged needs_manual_review: 1


3 — known edge case correction:

In [3]:
# Known edge case: row_id 63 has a blank transaction_ts. NaT sorts to the
# end of its entire (store, register) partition, which placed it next to
# an unrelated customer's transaction instead of its true siblings.
# invoice_number (INV-1009, shared with row_id 62 and 64) confirms where
# it actually belongs, so we correct it manually rather than guess further.
before_id = df.loc[df["row_id"] == 63, "reconstructed_txn_id"].iloc[0]
correct_id = df.loc[df["row_id"] == 62, "reconstructed_txn_id"].iloc[0]
df.loc[df["row_id"] == 63, "reconstructed_txn_id"] = correct_id

# check: confirm the correction actually applied, and that row 63 now
# shares an id with rows 62 and 64
after_id = df.loc[df["row_id"] == 63, "reconstructed_txn_id"].iloc[0]
print(f"row_id 63 reconstructed_txn_id: {before_id} -> {after_id}")
print(df[df["row_id"].isin([62, 63, 64])][["row_id", "customer_ref", "invoice_number", "reconstructed_txn_id"]])

row_id 63 reconstructed_txn_id: 46 -> 44
     row_id customer_ref invoice_number  reconstructed_txn_id
99       62         C003       INV-1009                    44
100      64         C003       INV-1009                    44
104      63         C003       INV-1009                    44


4 — build the transaction-level table:

In [4]:
transactions = df.groupby("reconstructed_txn_id").agg(
    store_id=("store_id", "first"),
    register_id=("register_id", "first"),
    customer_ref=("customer_ref", "first"),
    start_ts=("ts_parsed", "min"),
    end_ts=("ts_parsed", "max"),
    n_lines=("row_id", "count"),
    invoice_numbers_seen=("invoice_number", lambda s: sorted(set(x for x in s if isinstance(x, str) and x != ""))),
).reset_index()

review_flag = df.groupby("reconstructed_txn_id")["needs_manual_review"].any()
transactions["needs_manual_review"] = transactions["reconstructed_txn_id"].map(review_flag)

# check: one row per transaction, and the row count should be smaller
# than the original 131 line items
print("transactions table shape:", transactions.shape)
transactions.head()

transactions table shape: (63, 9)


,reconstructed_txn_id,store_id,register_id,customer_ref,start_ts,end_ts,n_lines,invoice_numbers_seen,needs_manual_review
0,1,ST01,REG-1,C003,2025-03-03 00:00:00,2025-03-03 00:00:00,4,[],False
1,2,ST01,REG-1,C003,2025-03-03 12:15:00,2025-03-03 12:16:00,3,[INV-1001],False
2,3,ST01,REG-1,NaN,2025-03-03 14:29:00,2025-03-03 14:29:00,2,[INV-1004],False
3,4,ST01,REG-1,C005,2025-03-03 17:08:38,2025-03-03 17:09:29,2,[],False
4,5,ST01,REG-1,C001,2025-03-03 22:18:00,2025-03-03 22:18:00,2,[],False


5 — final verification:

In [5]:
# invoice numbers that still map to more than 1 transaction —
# should only be INV-1000, which is a deliberate reused-number case, not a bug
has_invoice = df[df["invoice_number"].notna() & (df["invoice_number"] != "")]
invoice_check = has_invoice.groupby("invoice_number")["reconstructed_txn_id"].nunique().sort_values(ascending=False)
print("Invoices mapping to >1 transaction (expect only INV-1000):")
print(invoice_check[invoice_check > 1])

# transactions still flagged for manual review after the known correction
print("\nRemaining flagged transactions:")
print(transactions[transactions["needs_manual_review"]])

# row-count sanity check — these two numbers must match
print(f"\nTotal line items: {len(df)}")
print(f"Sum of n_lines across transactions: {transactions['n_lines'].sum()}")

Invoices mapping to >1 transaction (expect only INV-1000):
invoice_number
INV-1000    2
Name: reconstructed_txn_id, dtype: int64

Remaining flagged transactions:
    reconstructed_txn_id store_id register_id customer_ref  \
43                    44     ST99       REG-1         C003   

              start_ts              end_ts  n_lines invoice_numbers_seen  \
43 2025-03-03 18:35:00 2025-03-03 18:36:00        3           [INV-1009]   

    needs_manual_review  
43                 True  

Total line items: 131
Sum of n_lines across transactions: 131
